<a href="https://colab.research.google.com/github/epears04/data-4620-project/blob/main/mobileNet_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Mount Google Drive
Run the cell below to authorize Colab to access your Google Drive files to access CNR Park Dataset

In [6]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

### Step 2: Build a PyTorch MobileNetV2 Binary Classifier
The ImageNet-pretrained PyTorch MobileNetV2 backbone is adapted to predict the CNR-EXT occupancy labels: `0` (empty) or `1` (occupied).

In [ ]:
# !pip install -q torch torchvision tqdm scikit-learn

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Transfer-learning model for binary parking-occupancy labels (0 = empty, 1 = occupied).
weights = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weights)

# Train only the classifier head initially; this is much faster on Colab.
for parameter in model.features.parameters():
    parameter.requires_grad = False

model.classifier[1] = nn.Linear(model.last_channel, 1)
model = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)

### Step 3: Load CNR-EXT Patches and Official Splits from Google Drive
Upload the project folder to Google Drive and adjust `PROJECT_ROOT` if needed. The code reads the official `train`, `val`, and `test` text files, so it does not depend on class-named image folders.

In [ ]:
import os
import zipfile
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Upload this project folder to My Drive, preserving this structure:
# data-4620-project/
#   CNR-EXT-Patches-150x150/PATCHES/...
#   splits
PROJECT_ROOT = '/content/drive/MyDrive/data-4620-project'
DATA_ROOT = os.path.join(PROJECT_ROOT, 'CNR-EXT-Patches-150x150')
PATCHES_ROOT = os.path.join(DATA_ROOT, 'PATCHES')
SPLITS_ROOT = os.path.join(PROJECT_ROOT, 'splits', 'CNRPark-EXT')

# Extract once if only splits.zip was uploaded.
if not os.path.isdir(SPLITS_ROOT):
    splits_zip = os.path.join(PROJECT_ROOT, 'splits.zip')
    if not os.path.isfile(splits_zip):
        raise FileNotFoundError(f'Missing {splits_zip}')
    with zipfile.ZipFile(splits_zip) as archive:
        archive.extractall(PROJECT_ROOT)

# extract the patches once if only their ZIP file was uploaded.
if not os.path.isdir(PATCHES_ROOT):
    patches_zip = os.path.join(PROJECT_ROOT, 'CNR-EXT-Patches-150x150.zip')
    if not os.path.isfile(patches_zip):
        raise FileNotFoundError(f'Missing patch directory or archive: {PATCHES_ROOT}')
    with zipfile.ZipFile(patches_zip) as archive:
        archive.extractall(PROJECT_ROOT)

if not os.path.isdir(PATCHES_ROOT):
    raise FileNotFoundError(f'Patch directory was not found after extraction: {PATCHES_ROOT}')

BATCH_SIZE = 32
NUM_WORKERS = 2

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

class CNRDataset(Dataset):
    def __init__(self, split_name):
        split_file = os.path.join(SPLITS_ROOT, f'{split_name}.txt')
        with open(split_file) as file:
            self.samples = [
                (os.path.join(PATCHES_ROOT, path), float(label))
                for path, label in (line.strip().split() for line in file)
            ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        image = Image.open(image_path).convert('RGB')
        return transform(image), torch.tensor(label, dtype=torch.float32)

train_dataset = CNRDataset('train')
val_dataset = CNRDataset('val')
test_dataset = CNRDataset('test')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Datasets loaded: {len(train_dataset):,} train, {len(val_dataset):,} validation, {len(test_dataset):,} test')

### Step 4: Train and Evaluate with tqdm
Train the binary occupancy classifier, then evaluate it on the official held-out test split.

In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score

def evaluate(loader, description):
    model.eval()
    correct = total = 0
    all_labels, all_probabilities = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc=description):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            probabilities = torch.sigmoid(model(images).squeeze(1))
            predictions = (probabilities >= 0.5).float()

            correct += (predictions == labels).sum().item()
            total += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())

    return correct / total, roc_auc_score(all_labels, all_probabilities)

EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    progress = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{EPOCHS}')
    for images, labels in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        progress.set_postfix(loss=f'{loss.item():.4f}')

    val_accuracy, val_auc = evaluate(val_loader, 'Validation')
    print(f'Epoch {epoch + 1}: loss={running_loss / len(train_loader):.4f}, val_accuracy={val_accuracy:.2%}, val_auc={val_auc:.3f}')

test_accuracy, test_auc = evaluate(test_loader, 'Test')
print(f'Test accuracy: {test_accuracy:.2%}')
print(f'Test AUC: {test_auc:.3f}')